# Stage D1 — Metrics & Physical Consistency

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.6 (Eq. 3.21–3.27), feedback items on
metric justification, negative R², and residual analysis.

**What this stage is for:** everything since Phase A has been building
toward this comparison. B2 is the no-physics baseline; C4 is the
physics-informed ensemble. This notebook asks two separate questions,
deliberately kept separate: is the PINN *more accurate*, and is it
*more physically consistent* — a model can improve on one without the
other, and the text's own framing treats them as independent axes, not
one combined score.

**Lightweight by design:** unlike C1–C4, this stage doesn't train
anything — it loads B2's and C4's already-saved models and predictions
and evaluates them.

**Small-test-set caution, stated up front:** the test set has 6
points. R² on 6 points is noisy, a paired statistical test on 6 points
has very little power, and — as Sec. 3.6 itself warns — "good
performance on held-out data does not guarantee generalization... particularly
in regions not represented in the experimental dataset." Numbers below
are reported as computed, not oversold.

**Input:** `outputs/B2_predictions.csv`, `outputs/B2_baseline_model.keras`,
`outputs/C4_ensemble_predictions.csv`, `outputs/C4_ensemble_member_*.keras`,
`outputs/C1_collocation_points.csv`
**Output:** the accuracy/consistency comparison tables and figures,
plus an explicit gate verdict (PINN ≥ baseline on both axes?).


## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import wilcoxon
import tensorflow as tf
from tensorflow import keras

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42


## 1. Load data, saved models, and saved predictions

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

# same split/normalization logic as every earlier stage
df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

def split_arrays(split_name, cols):
    sub = df.filter(pl.col("split") == split_name)
    return sub.select([f"{c}_norm" for c in cols]).to_numpy().astype(np.float32)

X_train, Y_train = split_arrays("train", INPUT_COLS), split_arrays("train", OUTPUT_COLS)
X_val, Y_val = split_arrays("val", INPUT_COLS), split_arrays("val", OUTPUT_COLS)
X_test, Y_test = split_arrays("test", INPUT_COLS), split_arrays("test", OUTPUT_COLS)
X_trainval = np.concatenate([X_train, X_val]); Y_trainval = np.concatenate([Y_train, Y_val])
N_test = X_test.shape[0]

# saved models
def load_model_any_ext(stem):
    for ext in (".keras", ".h5"):
        p = OUT_DIR / f"{stem}{ext}"
        if p.exists():
            return keras.models.load_model(p, compile=False)
    raise FileNotFoundError(f"no {stem}.keras or {stem}.h5 in {OUT_DIR}")

baseline_model = load_model_any_ext("B2_baseline_model")
ensemble_models = [load_model_any_ext(f"C4_ensemble_member_{m}") for m in range(10)]
print(f"Loaded baseline + {len(ensemble_models)} ensemble members")

def ensemble_predict(X):
    preds = np.stack([m(X, training=False).numpy() for m in ensemble_models], axis=0)
    return preds.mean(axis=0)

pred_baseline_test = baseline_model(X_test, training=False).numpy()
pred_pinn_test = ensemble_predict(X_test)

colloc_df = pl.read_csv(OUT_DIR / "C1_collocation_points.csv")
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
print("test set:", X_test.shape[0], "points; collocation set:", X_colloc.shape[0], "points")


## 2. Accuracy metrics (Eq. 3.21–3.24)

$$
R^2_k = 1-\frac{\sum(y^{\text{pred}}_{i,k}-y^{\text{exp}}_{i,k})^2}{\sum(y^{\text{exp}}_{i,k}-\bar y_k)^2}, \quad
\text{RMSE}_k=\sqrt{\tfrac{1}{N}\sum(y^{\text{pred}}-y^{\text{exp}})^2}, \quad
\text{MAE}_k=\tfrac{1}{N}\sum|y^{\text{pred}}-y^{\text{exp}}|
$$

$$
\text{MAPE}_k = \frac{100\%}{N}\sum\left|\frac{y^{\text{pred}}_{i,k}-y^{\text{exp}}_{i,k}}{y^{\text{exp}}_{i,k}}\right|
$$

**MAPE caveat, checked below before trusting it:** the formula divides
by the true value itself — any test point where an output is near zero
(normalized space makes this a real possibility, e.g. an eta point
near its train-min) inflates MAPE arbitrarily. Flagged explicitly
rather than silently reported.

In [ ]:
def compute_metrics(pred, truth):
    rows = []
    for k, out in enumerate(OUTPUT_COLS):
        p, y = pred[:, k], truth[:, k]
        ss_res = np.sum((y - p) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        rmse = np.sqrt(np.mean((p - y) ** 2))
        mae = np.mean(np.abs(p - y))
        near_zero = np.abs(y) < 1e-3
        if near_zero.any():
            mape = float("nan")
        else:
            mape = 100 * np.mean(np.abs((p - y) / y))
        rows.append({"output": out, "r2": r2, "rmse": rmse, "mae": mae, "mape": mape,
                     "mape_unreliable_near_zero": bool(near_zero.any())})
    return pl.DataFrame(rows)

metrics_baseline = compute_metrics(pred_baseline_test, Y_test)
metrics_pinn = compute_metrics(pred_pinn_test, Y_test)
print("Baseline:"); print(metrics_baseline)
print("\nPINN ensemble:"); print(metrics_pinn)

neg_r2_baseline = metrics_baseline.filter(pl.col("r2") < 0)["output"].to_list()
neg_r2_pinn = metrics_pinn.filter(pl.col("r2") < 0)["output"].to_list()
if neg_r2_baseline:
    print(f"\nNegative R2, baseline: {neg_r2_baseline} -- worse than predicting the mean on 6 test points.")
if neg_r2_pinn:
    print(f"Negative R2, PINN: {neg_r2_pinn} -- worse than predicting the mean on 6 test points.")


## 3. Accuracy comparison, visually

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=["R2 (higher better)", "RMSE (lower better)",
                                                          "MAE (lower better)", "MAPE % (lower better)"])
metric_names = ["r2", "rmse", "mae", "mape"]
for i, m in enumerate(metric_names):
    r, c = divmod(i, 2)
    fig.add_trace(go.Bar(x=OUTPUT_COLS, y=metrics_baseline[m], name="baseline",
                          marker_color="#993C1D", showlegend=(i == 0)), row=r + 1, col=c + 1)
    fig.add_trace(go.Bar(x=OUTPUT_COLS, y=metrics_pinn[m], name="PINN",
                          marker_color="#185FA5", showlegend=(i == 0)), row=r + 1, col=c + 1)
fig.update_layout(barmode="group", height=650, width=900, title_text="Accuracy: baseline vs. PINN (test set, n=6)")
fig.show()


## 4. Physical consistency metrics (Eq. 3.25–3.27 + non-negativity)

$$
\text{MSR}_j=\frac{1}{N_{\text{eval}}}\sum I\!\left(s_j\tfrac{\partial y}{\partial x}\Big|_i \le 0\right), \quad
\text{TSC}=\frac{1}{N_{\text{eval}}}\sum I\!\left(\tfrac{\partial y_1}{\partial x}\cdot\tfrac{\partial y_2}{\partial x} < 0\right), \quad
\text{CSR}=\frac{1}{N_{\text{eval}}}\sum I\!\left(\tfrac{\partial^2 y}{\partial x^2}\Big|_i \ge 0\right)
$$

Evaluated on the 1000 collocation points ("throughout the domain",
Sec. 3.6.2's own phrase), not just the 6 test points — six points
would give a rate metric with only 7 possible values (0/6...6/6), too
coarse to be informative. The PINN ensemble is evaluated as its **mean
prediction function** (averaging predictions first, then
differentiating — equivalent by linearity to averaging each member's
own derivative).

In [ ]:
def monotonic_indicator(predict_fn, x, out_idx, in_idx, sign=1.0):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx].numpy()
    return (sign * d <= 0), d

def tradeoff_indicator(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    da = tape.gradient(a, x_t)[:, in_idx].numpy()
    db = tape.gradient(b, x_t)[:, in_idx].numpy()
    del tape
    return (da * db < 0), da * db

def convexity_indicator(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d1 = tape1.gradient(target, x_t)[:, in_idx]
    d2 = tape2.gradient(d1, x_t)[:, in_idx].numpy()
    return (d2 >= 0), d2

def mean_positive_violation(signed_quantity):
    violating = signed_quantity[signed_quantity > 0]
    return float(violating.mean()) if len(violating) else 0.0

def nonneg_compliance(predict_fn, x, out_idxs=EMISSION_IDXS):
    y = predict_fn(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()
    emissions = y[:, out_idxs]
    compliant = emissions >= 0
    return compliant.mean(), np.where(emissions < 0, -emissions, 0)

def evaluate_consistency(predict_fn, label):
    rows = []
    ind, d = monotonic_indicator(predict_fn, X_colloc, NOX_IDX, SOI_IDX, sign=1.0)
    rows.append({"model": label, "constraint": "NOx-SOI (MSR)", "rate": ind.mean(),
                 "mean_violation": mean_positive_violation(d)})
    ind, d = monotonic_indicator(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX, sign=1.0)
    rows.append({"model": label, "constraint": "PM-lambda (MSR)", "rate": ind.mean(),
                 "mean_violation": mean_positive_violation(d)})
    ind, d2 = convexity_indicator(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX)
    rows.append({"model": label, "constraint": "HC-lambda (CSR)", "rate": ind.mean(),
                 "mean_violation": mean_positive_violation(-d2)})
    ind, prod = tradeoff_indicator(predict_fn, X_colloc, NOX_IDX, PM_IDX, SOI_IDX)
    rows.append({"model": label, "constraint": "NOx-PM (TSC)", "rate": ind.mean(),
                 "mean_violation": mean_positive_violation(prod)})
    ind, prod = tradeoff_indicator(predict_fn, X_colloc, ETA_IDX, NOX_IDX, SOI_IDX)
    rows.append({"model": label, "constraint": "eta-NOx (TSC)", "rate": ind.mean(),
                 "mean_violation": mean_positive_violation(prod)})
    rate, viol = nonneg_compliance(predict_fn, X_colloc)
    rows.append({"model": label, "constraint": "non-negativity", "rate": rate,
                 "mean_violation": float(viol[viol > 0].mean()) if (viol > 0).any() else 0.0})
    return rows

consistency_rows = (
    evaluate_consistency(lambda x: baseline_model(x, training=False), "baseline")
    + evaluate_consistency(lambda x: tf.reduce_mean(tf.stack([m(x, training=False) for m in ensemble_models], axis=0), axis=0), "PINN")
)
consistency_df = pl.DataFrame(consistency_rows)
consistency_df


## 5. Consistency comparison, visually

In [ ]:
pivot_rows = []
for constraint in consistency_df["constraint"].unique().to_list():
    row = {"constraint": constraint}
    for model in ["baseline", "PINN"]:
        val = consistency_df.filter((pl.col("constraint") == constraint) & (pl.col("model") == model))["rate"]
        row[model] = val[0] if len(val) else None
    pivot_rows.append(row)
pivot_df = pl.DataFrame(pivot_rows)

fig = go.Figure()
fig.add_trace(go.Bar(x=pivot_df["constraint"], y=pivot_df["baseline"], name="baseline", marker_color="#993C1D"))
fig.add_trace(go.Bar(x=pivot_df["constraint"], y=pivot_df["PINN"], name="PINN", marker_color="#185FA5"))
fig.update_layout(barmode="group", title="Physical consistency rate, evaluated on 1000 collocation points",
                   yaxis_title="satisfaction rate", yaxis_range=[0, 1.02], width=850, height=450)
fig.show()


## 6. AIC (feedback: weigh error against model complexity)

Not part of Eq. 3.21–3.27 — added because the feedback specifically
asked for a metric that trades off accuracy against complexity.

$$
\text{AIC} = n\ln(\text{RSS}/n) + 2k
$$

**Computed on the 34-point train+val pool, not the 6-point test
set — deliberately.** AIC's small-sample correction (AICc) adds a
$\frac{2k(k+1)}{n-k-1}$ term; with $k$ in the dozens-to-low-hundreds
(these networks' parameter counts) and $n=6$, $n-k-1$ is negative,
so AICc is undefined on the test set — not just imprecise, mathematically
broken. 34 points doesn't fully fix this but keeps $n-k-1$ meaningfully
positive for the smaller architectures at least.

In [ ]:
def aic(pred, truth, k_params):
    residuals = (pred - truth).ravel()
    n_obs = residuals.size
    rss = np.sum(residuals ** 2)
    return n_obs * np.log(rss / n_obs) + 2 * k_params

pred_baseline_tv = baseline_model(X_trainval, training=False).numpy()
pred_pinn_tv = ensemble_predict(X_trainval)

k_baseline = baseline_model.count_params()
k_pinn = ensemble_models[0].count_params()  # all 10 members share the same architecture

aic_table = pl.DataFrame([
    {"model": "baseline", "k_params": k_baseline, "AIC": aic(pred_baseline_tv, Y_trainval, k_baseline)},
    {"model": "PINN (1 member's k, ensemble-mean residuals)", "k_params": k_pinn,
     "AIC": aic(pred_pinn_tv, Y_trainval, k_pinn)},
])
aic_table


## 7. Paired statistical comparison

Wilcoxon signed-rank on per-point squared error, same 6 test points
for both models (paired, since it's the same points) — **treat this as
a weak, exploratory signal, not a verdict**: n=6 gives a paired test
very little power, consistent with Sec. 3.6's own caution about this
dataset size.

In [ ]:
rows = []
for k, out in enumerate(OUTPUT_COLS):
    se_baseline = (pred_baseline_test[:, k] - Y_test[:, k]) ** 2
    se_pinn = (pred_pinn_test[:, k] - Y_test[:, k]) ** 2
    try:
        stat, p = wilcoxon(se_baseline, se_pinn)
    except ValueError:
        p = float("nan")  # all differences zero/tied -- can happen with n=6
    rows.append({"output": out, "p_value": p,
                 "pinn_better_on_average": bool(se_pinn.mean() < se_baseline.mean())})
pl.DataFrame(rows)


## 8. Residual analysis (vs. each input, both models)

In [ ]:
fig = make_subplots(rows=5, cols=4, subplot_titles=[f"{o} vs {i}" for o in OUTPUT_COLS for i in INPUT_COLS])
for oi, out in enumerate(OUTPUT_COLS):
    resid_baseline = pred_baseline_test[:, oi] - Y_test[:, oi]
    resid_pinn = pred_pinn_test[:, oi] - Y_test[:, oi]
    for ii, inp in enumerate(INPUT_COLS):
        r, c = oi + 1, ii + 1
        fig.add_trace(go.Scatter(x=X_test[:, ii], y=resid_baseline, mode="markers",
                                  marker=dict(color="#993C1D", size=7), name="baseline",
                                  showlegend=(oi == 0 and ii == 0)), row=r, col=c)
        fig.add_trace(go.Scatter(x=X_test[:, ii], y=resid_pinn, mode="markers",
                                  marker=dict(color="#185FA5", size=7), name="PINN",
                                  showlegend=(oi == 0 and ii == 0)), row=r, col=c)
        fig.add_hline(y=0, line_color="#B0AFA8", line_width=1, row=r, col=c)
fig.update_layout(height=1150, width=950, title_text="Residuals (pred - actual) vs. each input, test set")
fig.show()


## 9. Gate verdict — does the PINN clear B1's original bar?

In [ ]:
acc_wins = sum(1 for k in range(N_OUT)
                if metrics_pinn["rmse"][k] <= metrics_baseline["rmse"][k])
cons_wins = sum(1 for c in pivot_df["constraint"].to_list()
                 if pivot_df.filter(pl.col("constraint") == c)["PINN"][0] >=
                    pivot_df.filter(pl.col("constraint") == c)["baseline"][0])

print(f"Accuracy: PINN RMSE <= baseline on {acc_wins}/{N_OUT} outputs")
print(f"Consistency: PINN rate >= baseline on {cons_wins}/{len(pivot_df)} constraints")
print()
if acc_wins >= N_OUT // 2 + 1 and cons_wins >= len(pivot_df) // 2 + 1:
    print("GATE: PASS (on majority of both axes) -- but re-read Sections 2 and 7 before")
    print("writing this up as a clean win; n=6 accuracy numbers and a low-power paired")
    print("test both deserve the same caution the dissertation text itself asks for.")
else:
    print("GATE: NOT a clean pass on both axes -- check whether accuracy or consistency")
    print("(or a specific output/constraint) is the weak point before concluding physics")
    print("didn't help; a mixed result on 6 test points is not the same as a negative result.")


## Optional — persist outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_baseline.write_csv(OUT_DIR / "D1_accuracy_baseline.csv")
metrics_pinn.write_csv(OUT_DIR / "D1_accuracy_pinn.csv")
consistency_df.write_csv(OUT_DIR / "D1_consistency.csv")
aic_table.write_csv(OUT_DIR / "D1_aic.csv")
print(f"Saved to {OUT_DIR}")


## Next

**D2** (uncertainty via MC Dropout) and **E1/E2** (optimization) both
consume the PINN ensemble this notebook already validated — if the
gate above didn't pass cleanly, it's worth deciding whether to proceed
to optimization anyway (with the caveats documented) or revisit C3's
search (e.g. excluding `relu`, per C3.1) before continuing.
